In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/silver_utils

In [0]:
# Cell 1: Setup, Context & Environment Initialization
db_helpers = DatabricksHelpers(dbutils)
current_user_email = db_helpers.current_user()

catalog_name = "f1_dev"
schemas = ["bronze", "silver", "gold"]

# Ensure bronze, silver, and gold schemas exist in Unity Catalog
initialize_environment(spark, catalog_name, schemas)

source_table = f"{catalog_name}.bronze.races"
target_table = f"{catalog_name}.silver.dim_races"

# --- CELL 1 TEST ---
assert current_user_email and ("@" in current_user_email or current_user_email == "root"), \
    f"TEST FAILED: Invalid user context derived: '{current_user_email}'"

assert spark.catalog.tableExists(source_table), \
    f"TEST FAILED: Source bronze table '{source_table}' does not exist in catalog."

print(f"✓ Cell 1 Passed: Executed by '{current_user_email}' on catalog '{catalog_name}'.")

In [0]:
# Cell 2: Data Extraction & Bronze String Sanitization
df_bronze = spark.table(source_table)

# Reusable string cleaning: purges '\N', 'NULL', 'null', and empty strings across all text fields
df_sanitized = sanitize_bronze_strings(df_bronze)

# --- CELL 2 TEST ---
string_cols = [f.name for f in df_sanitized.schema.fields if f.dataType.typeName() == "string"]
for col_name in string_cols:
    dirty_count = df_sanitized.filter(
        (F.col(col_name) == r"\N") | (F.lower(F.col(col_name)) == "null")
    ).count()
    assert dirty_count == 0, f"TEST FAILED: Found {dirty_count} unhandled string literals in '{col_name}'"

print(f"✓ Cell 2 Passed: String sanitization verified across {len(string_cols)} string columns.")

In [0]:
# Cell 3: Silver Domain Transformations & Safe Date Casting
df_races_silver = (
    df_sanitized
    .select(
        F.col("raceId").try_cast("integer").alias("race_id"),
        F.col("year").try_cast("integer").alias("race_year"),
        F.col("round").try_cast("integer").alias("race_round"),
        F.col("circuitId").try_cast("integer").alias("circuit_id"),
        F.col("name").alias("race_name"),
        F.to_date(F.col("date"), "yyyy-MM-dd").alias("race_date")
    )
    # Deduplicate against primary key
    .dropDuplicates(["race_id"])
)

# --- CELL 3 TEST ---
sample_row = df_races_silver.first()

# 1. Primary & Foreign key data type verification
assert dict(df_races_silver.dtypes)["race_id"] == "int", "TEST FAILED: race_id cast failed."
assert dict(df_races_silver.dtypes)["circuit_id"] == "int", "TEST FAILED: circuit_id cast failed."

# 2. Date data type verification
assert dict(df_races_silver.dtypes)["race_date"] == "date", "TEST FAILED: race_date cast failed."

# 3. Confirm raw URL field was dropped
prohibited_cols = {"url"}
assert not prohibited_cols.intersection(set(df_races_silver.columns)), \
    f"TEST FAILED: Unneeded columns detected in Silver projection: {prohibited_cols.intersection(set(df_races_silver.columns))}"

print("✓ Cell 3 Passed: Race transformations, date casts, and column projections verified.")

In [0]:
# Cell 4: Pre-Write Data Quality Circuit Breaker & Domain Rules
# 1. Standard Circuit Breaker: Non-Null PK & Key Uniqueness assertion
run_circuit_breaker(
    df=df_races_silver, 
    primary_keys=["race_id"], 
    prohibited_pii={"url"}
)

# 2. Race Domain Integrity Assertions
null_dates = df_races_silver.filter(F.col("race_date").isNull()).count()
assert null_dates == 0, f"CIRCUIT BREAKER: Found {null_dates} records with unparseable or NULL race_date!"

invalid_years = df_races_silver.filter((F.col("race_year") < 1950) | (F.col("race_year") > 2030)).count()
assert invalid_years == 0, f"CIRCUIT BREAKER: Found {invalid_years} records with invalid race_year (< 1950 or > 2030)!"

invalid_rounds = df_races_silver.filter(F.col("race_round") <= 0).count()
assert invalid_rounds == 0, f"CIRCUIT BREAKER: Found {invalid_rounds} records with race_round <= 0!"

print("✓ Cell 4 Passed: Pre-write circuit breaker and race domain assertions passed.")

In [0]:
# Cell 5: Target Delta DDL Creation & Idempotent Delta Load
# 1. Ensure target Silver Dimension Delta table exists with Primary Key constraint
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        race_id INT NOT NULL,
        race_year INT,
        race_round INT,
        circuit_id INT,
        race_name STRING,
        race_date DATE,
        CONSTRAINT dim_races_pk PRIMARY KEY (race_id)
    )
    USING DELTA
""")

# 2. Reusable Upsert: Enables CDF, Auto-Optimize, and executes SCD Type 1 MERGE
upsert_silver_delta(
    spark=spark, 
    df=df_races_silver, 
    target_table=target_table, 
    primary_keys=["race_id"]
)

# --- CELL 5 TEST ---
target_df = spark.table(target_table)
staging_count = df_races_silver.count()
target_count = target_df.count()

# Confirm written count matches staging expectations
assert target_count >= staging_count, \
    f"TEST FAILED: Target table count ({target_count}) is lower than staging count ({staging_count})."

print(f"✓ Cell 5 Passed: Delta MERGE load to '{target_table}' completed and verified.")

In [0]:
# Cell 6: Unity Catalog Metadata Governance & Exit
# Reusable Governance Tagger: Sets tags and asserts existence in information_schema
apply_silver_tags(
    spark=spark, 
    active_catalog=catalog_name, 
    target_table=target_table, 
    current_user_email=current_user_email, 
    table_type="dimension"
)

# Clean exit for Databricks Workflows / Jobs Orchestration
dbutils.notebook.exit(f"SUCCESS: Pipeline executed cleanly by {current_user_email} on {target_table}")